Abbiamo discusso come BIM sia limitato dal fatto che non prende in considerazione la term frequency e la lunghezza del documento.

Ricordiamo la formula generale che avevamo ricavato con BIM:
$$
RSV_d =
\log \prod_{t_i:\,x_i=y_i=1}
\frac{
Pr(x_i = 1 \mid R, v_q)\,Pr(x_i = 0 \mid \bar R, v_q)
}{
Pr(x_i = 1 \mid \bar R, v_q)\,Pr(x_i = 0 \mid R, v_q)
}
$$
Usando l'assunzione per cui i documenti rilevanti sono una piccola frazione della collezione, possiamo approssimare il comportamento dei documenti non rilevanti con quello dell’intera collezione:

$$
Pr(x_i = 1 \mid \bar R, v_q) \approx Pr(x_i=1)
$$

$$
Pr(x_i = 0 \mid \bar R, v_q) \approx Pr(x_i=0)
$$

ottenendo:

$$
RSV_d =
\sum_{t_i:\,x_i=y_i=1}
\log
\frac{
Pr(x_i = 1 \mid R, v_q)\,Pr(x_i=0)
}{
Pr(x_i = 1)\,Pr(x_i = 0 \mid R, v_q)
}
$$

### Passaggio da BIM a modello con TF
Nel BIM classico per ogni termine i si ha solo $x_i \in \{0,1\}$, indicando se il termine è presente o meno nel documento. Tuttavia, questo approccio non tiene conto di quanto spesso un termine appare in un documento (term frequency). 

Per introdurre la **term frequency TF**, si sostituisce quindi la variabile binaria $x_i$ con una variabile aleatoria di conteggio $d_i =$ #occorrenze del termine $i$ in $d$. Non mi interessa quindi più solo se il termine è presente o meno, ma voglio modellare quante volte compare.

Quindi la formula cambia: 
$$ RSV_d = \sum_{t_i:\,x_i=y_i=1} \log \frac{Pr(d_i = n_i \mid R, v_q) Pr(d_i=0)}{Pr(d_i = n_i) Pr(d_i = 0 \mid \bar R, v_q)} $$

Per utilizzare la formula quindi dobbiamo scegliere un modello probabilistico per i conteggi, cioè dobbiamo modellare la variabile aleatoria $d_i$. 

Poiché è una v.a. discreta e si vogliono modellare conteggi del tipo $Pr(d_i = k)$ (probabilità che il termine $i$ appaia $k$ volte in un documento), una prima idea è utilizzare una **Binomiale**.

In questo senso si pensa a un documento come un insieme di $l$ posizioni (parole), e **in ogni posizione il termine compare con una certa probabilità $\tilde{p}$. In tal caso $d_i \sim \mathrm{Binomial}(l, \tilde{p})$** e quindi:
$$Pr(d_i = k) = \binom{l}{k} \tilde{p}^k (1-\tilde{p})^{l-k}$$

Tuttavia la distribuzione binomiale ha alcuni problemi: **dipende da l (lunghezza del documento)**, ha formule brutte ed in generale è difficile da usare nel ranking. Per questo motivo si preferisce fare affidamento a un trick matematico e passare alla **Poisson**.

Quando $l$ è grande e $\tilde{p}$ è piccolo, la distribuzione binomiale può essere approssimata con una distribuzione di Poisson di parametro $\lambda = l\tilde{p}$:

$$
\mathrm{Binomial}(l,\tilde{p}) \approx \mathrm{Poisson}(\lambda)
$$

con:

$$
\lambda = \text{valore atteso di occorrenze del termine} = l\tilde{p}
$$
Poisson è migliore della binomiale poiché ha una formula più semplice da maneggiare. Inoltre intuitivamente $\lambda$ è grande se il termine è frequente e piccolo se è raro, quindi è un buon parametro per modellare la frequenza dei termini.

Quindi per stimare la $Pr(d_i = x)$ (il termine $i$ appare $x$ volte in un documento) si avrà:
$$
Pr(d_i = x) = \frac{\lambda^x e^{-\lambda}}{x!}
$$

#### 2-Poisson 
Ma come stimare $\lambda$? Una prima idea ragionevole è $$\lambda_j \approx \frac{cf_j}{N}$$ cioè il numero atteso di occorrenze del termine $j$ in un documento è dato approssimativamente dal numero totale di occorrenze del termine $j$ nella collezione ($cf_j$) diviso il numero totale di documenti ($N$).

Tuttavia ci rendiamo conto che **non basta una sola $\lambda$, poiché un termine si comporta diversamente in base al fatto che appartenga a documenti rilevanti o alla collezione in generale** 

(Ossia la frequenza attesa di un termine può cambiare a seconda del tipo di documento che sto guardando: ad es. data la query "health plan", il termine $t_j = \text{health}$ potrebbe apparire in tutta la collezione mediamente poco (es. $\gamma_j = 0.2$), mentre nei documenti rilevanti per la query molto più spesso (es. $\rho_j = 3$ ossia 3volte per documento)).

Per questo motivo si introduce il modello **2-Poisson** che prevede due distribuzioni di Poisson distinte per i documenti rilevanti e non rilevanti, con parametri:
- $\rho_j$ = frequenza attesa del termine $j$ nei documenti rilevanti
- $\gamma_j$ = frequenza attesa del termine $j$ nell'intera collezione

Usando questo modello otteniamo che:
$$Pr(d_i = n_i \mid R, v_q) = \frac{\rho_j^{n_i} e^{-\rho_j}}{n_i!}$$
$$Pr(d_i = 0 \mid R, v_q) = e^{-\rho_j}$$
$$Pr(d_i = n_i) = \frac{\gamma_j^{n_i} e^{-\gamma_j}}{n_i!}$$
$$Pr(d_i = 0 \mid \bar R, v_q) = e^{-\gamma_j}$$

Sostituendo nella formula del RSV otteniamo:
$$RSV_d = \sum_{t_i:\,x_i=y_i=1} \log \frac{\frac{\rho_j^{n_i} e^{-\rho_j}}{n_i!} e^{-\gamma_j}}{\frac{\gamma_j^{n_i} e^{-\gamma_j}}{n_i!} e^{-\rho_j}} = \sum_{t_i:\,x_i=y_i=1} \log \frac{\rho_j^{n_i}}{\gamma_j^{n_i}} = \sum_{t_i:\,x_i=y_i=1} n_i \log \frac{\rho_j}{\gamma_j}$$

Analizziamo la formula ottenuta: abbiamo una sommatoria per ogni termine in comune tra query e documento di:
- $n_i$ = quante volte il termine $i$ comprare -> ogni occorrenza del termine aumenta il punteggio
- Il peso $log \frac{\rho_j}{\gamma_j}$: rappresenta il rapporto in log tra la frequenza attesa del termine nei rilevanti e la frequenza attesa del termine in generale. Se $\rho_j$ è molto più grande di $\gamma_j$ (il termine è molto più frequente nei rilevanti che in generale) allora il peso è alto, altrimenti se appare più o meno con la stessa frequenza nei rilevanti e in generale allora il rapporto è vicino a 1 e quindi log vicino a 0.

Abbiamo quindi ottenuto una formula del tipo $RSV_d = \sum_{t_i:\,x_i=y_i=1} n_i w_i$, cioè **term frequency (n_i) per un peso w_i che ricorda concettualmente idf (infatti quel rapporto misura quanto un termine è più "speciale" nei documenti rilevanti rispetto al suo comportamento normale nella collezione, con idf similmente il termine pesa tanto se compare in pochi documenti nella collezione)**. Stiamo quindi in un certo senso ricostruendo tf-idf in modo probabilistico, partendo da un modello di base (BIM) e introducendo la term frequency con il modello 2-Poisson.

**PROBLEMA**: $\gamma_j$ come visto può essere stimato come $\gamma_j \approx \frac{cf_j}{N}$, ma **per stimare $\rho_j$ (frequenza attesa del termine nei rilevanti) serve sapere quali documenti sono rilevanti per la query, ma questo è esattamente quello che stiamo cercando a priori.** Quindi questo modello non è direttamente utilizzabile.